# Task2: Imlement the decision tree from scratch

In [1]:
import importlib
import json
import os
import pickle
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from pathlib import Path

# Default directory for evaluate_three_models_one_fold_preselected checkpoints (override if needed).
THREE_MODEL_CHECKPOINT_ROOT = Path.cwd() / "checkpoints_three_models"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo

import dtree_process_worker as _dtree_worker

# Force refresh in notebooks so newly added helpers are visible.
_dtree_worker = importlib.reload(_dtree_worker)

if hasattr(_dtree_worker, "build_class_folds_worker"):
    build_class_folds_worker = _dtree_worker.build_class_folds_worker
else:
    # Fallback keeps notebook runnable even if module is stale.
    def build_class_folds_worker(args):
        class_value, class_indices, k, seed = args
        indices = np.asarray(class_indices, dtype=np.int64).copy()
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)
        split_indices = np.array_split(indices, int(k))
#         return class_value, [chunk.astype(np.int64) for chunk in split_indices]


In [2]:
def entropy_from_labels(labels):
    labels_arr = np.asarray(labels, dtype=np.int64)
    if labels_arr.size == 0:
        return 0.0
    _, counts = np.unique(labels_arr, return_counts=True)
    probs = counts / counts.sum()
    return float(-np.sum(probs * np.log2(probs)))


def majority_label(labels):
    labels_arr = np.asarray(labels, dtype=np.int64)
    values, counts = np.unique(labels_arr, return_counts=True)
    return int(values[np.argmax(counts)])


def split_dataset_np(dataset, feature_index, feature_value):
    data = np.asarray(dataset)
    if data.size == 0:
        return data
    mask = data[:, feature_index] == feature_value
    filtered = data[mask]
    if filtered.size == 0:
        return np.empty((0, data.shape[1] - 1), dtype=data.dtype)
    return np.delete(filtered, feature_index, axis=1)


def _feature_gain_worker(args):
    dataset, feature_index, base_entropy = args
    data = np.asarray(dataset)
    values = np.unique(data[:, feature_index])
    total = float(data.shape[0])
    conditional_entropy = 0.0

    for value in values:
        subset = split_dataset_np(data, feature_index, value)
        if subset.shape[0] == 0:
            continue
        conditional_entropy += (subset.shape[0] / total) * entropy_from_labels(subset[:, -1])

    return feature_index, base_entropy - conditional_entropy


def best_feature_threaded(dataset, thread_workers=8):
    data = np.asarray(dataset)
    n_features = data.shape[1] - 1
    if n_features <= 1:
        return 0

    base_entropy = entropy_from_labels(data[:, -1])
    args = np.empty(n_features, dtype=object)
    for idx in range(n_features):
        args[idx] = (data, idx, base_entropy)

    workers = max(1, min(int(thread_workers), n_features))
    with ThreadPoolExecutor(max_workers=workers) as executor:
        results = tuple(executor.map(_feature_gain_worker, args))

    return int(max(results, key=lambda item: item[1])[0])


def build_tree(dataset, feature_names, min_sample_size=1, max_depth=None, depth=0, thread_workers=8):
    data = np.asarray(dataset)
    names = np.asarray(feature_names, dtype=object)

    labels = data[:, -1]

    # stop condition1: when number of samples less then setted
    if np.unique(labels).size <= max(1, min_sample_size):
        return majority_label(labels)

    # stop condition2: run out of features
    if data.shape[1] == 1:
        return majority_label(labels)

    # stop condition3: reach setted depth
    if max_depth is not None and depth >= max_depth:
        return majority_label(labels)

    

    best_feature = best_feature_threaded(data, thread_workers=thread_workers)
    root_name = names[best_feature]
    tree = {root_name: {}}

    child_feature_names = np.delete(names, best_feature)
    unique_values = np.unique(data[:, best_feature])
    for value in unique_values:
        child_data = split_dataset_np(data, best_feature, value)
        if child_data.shape[0] == 0:
            tree[root_name][int(value)] = majority_label(labels)
        else:
            tree[root_name][int(value)] = build_tree(
                child_data,
                child_feature_names,
                min_sample_size = min_sample_size,
                max_depth=max_depth,
                depth=depth + 1,
                thread_workers=thread_workers,
            )

    return tree


def predict_one(tree, feature_names, sample, default_label):
    node = tree
    names = np.asarray(feature_names, dtype=object)
    x = np.asarray(sample)

    while isinstance(node, dict):
        root = next(iter(node))
        children = node[root]

        matched = np.where(names == root)[0]
        if matched.size == 0:
            return default_label

        idx = int(matched[0])
        value = int(x[idx])
        if value not in children:
            return default_label

        node = children[value]
        names = np.delete(names, idx)
        x = np.delete(x, idx)

    return int(node)


def predict_batch(tree, feature_names, dataset, default_label):
    data = np.asarray(dataset)
    x = data[:, :-1]
    preds = np.empty(x.shape[0], dtype=np.int64)
    for i in range(x.shape[0]):
        preds[i] = predict_one(tree, feature_names, x[i], default_label)
    return preds


def accuracy_np(y_true, y_pred):
    true_arr = np.asarray(y_true, dtype=np.int64)
    pred_arr = np.asarray(y_pred, dtype=np.int64)
    if true_arr.size == 0:
        return 0.0
    return float(np.mean(true_arr == pred_arr))


def post_prune_tree_reduced_error(tree, train_data, valid_data, feature_names):
    if not isinstance(tree, dict):
        return tree

    train_np = np.asarray(train_data)
    valid_np = np.asarray(valid_data)
    names = np.asarray(feature_names, dtype=object)

    if train_np.shape[0] == 0:
        return tree

    root = next(iter(tree))
    children = tree[root]
    root_idx = int(np.where(names == root)[0][0])
    child_names = np.delete(names, root_idx)

    pruned_children = {}
    for edge_val, child in children.items():
        child_train = split_dataset_np(train_np, root_idx, edge_val)
        child_valid = split_dataset_np(valid_np, root_idx, edge_val)
        pruned_children[edge_val] = post_prune_tree_reduced_error(child, child_train, child_valid, child_names)

    pruned_tree = {root: pruned_children}

    if valid_np.shape[0] == 0:
        return pruned_tree

    default = majority_label(train_np[:, -1])
    subtree_pred = predict_batch(pruned_tree, names, valid_np, default)
    subtree_acc = accuracy_np(valid_np[:, -1], subtree_pred)

    leaf_pred = np.full(valid_np.shape[0], default, dtype=np.int64)
    leaf_acc = accuracy_np(valid_np[:, -1], leaf_pred)

    if leaf_acc >= subtree_acc:
        return int(default)

    return pruned_tree

In [3]:
def count_leaf_nodes(tree):
    if not isinstance(tree, dict):
        return 1
    root = next(iter(tree))
    return int(sum(count_leaf_nodes(child) for child in tree[root].values()))


def tree_depth(tree):
    if not isinstance(tree, dict):
        return 1
    root = next(iter(tree))
    child_depths = np.fromiter(
        (tree_depth(child) for child in tree[root].values()),
        dtype=np.int64,
    )
    return int(1 + child_depths.max(initial=0))


def save_markdown_report(results_array, output_path):
    rows = np.asarray(results_array, dtype=object)
    lines = np.empty(rows.shape[0] + 8, dtype=object)
    lines[0] = "# Fold Results"
    lines[1] = ""
    lines[2] = "| Fold | Unpruned Acc | Pruned Acc | Leaves | Depth |"
    lines[3] = "|---:|---:|---:|---:|---:|"

    unpruned = np.empty(rows.shape[0], dtype=np.float64)
    pruned = np.empty(rows.shape[0], dtype=np.float64)

    for i, item in enumerate(rows):
        unpruned[i] = float(item["unpruned_accuracy"])
        pruned[i] = float(item["pruned_accuracy"])
        lines[i + 4] = (
            f"| {item['fold']} | {unpruned[i]:.4f} | {pruned[i]:.4f} | "
            f"{item['leaf_count']} | {item['depth']} |"
        )

    lines[-4] = ""
    lines[-3] = f"- Mean unpruned accuracy: {unpruned.mean():.4f}"
    lines[-2] = f"- Mean pruned accuracy: {pruned.mean():.4f}"
    lines[-1] = ""

    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    output_file.write_text("\n".join(lines.tolist()), encoding="utf-8")
    print(f"Saved report: {output_file}")

# Task1: Loading dataset:
 1) Letter Recognition
 2) Adult
 3) Mushroom

## Missing value strategy
 1) calculate value distribution in which column missing value located in
 2) based on this value distribution, randomly sampling a value for missing one

## Binning 
 1) did binning for all numeric value in three dataset

## Install once if needed:
## %pip install ucimlrepo

In [4]:
# UCI IDs: Letter=59, Adult=2, Mushroom=73
letter_df = fetch_ucirepo(id=59).data.original.copy()
adult_df = fetch_ucirepo(id=2).data.original.copy()
mushroom_df = fetch_ucirepo(id=73).data.original.copy()

print(letter_df.shape)
print("Letter columns:", letter_df.columns.to_numpy(dtype=object))
print(adult_df.shape)
print("Adult columns:", adult_df.columns.to_numpy(dtype=object))
print(mushroom_df.shape)
print("mushroom columns:", mushroom_df.columns.to_numpy(dtype=object))
mushroom_df.head(5)

(20000, 17)
Letter columns: ['lettr' 'x-box' 'y-box' 'width' 'high' 'onpix' 'x-bar' 'y-bar' 'x2bar'
 'y2bar' 'xybar' 'x2ybr' 'xy2br' 'x-ege' 'xegvy' 'y-ege' 'yegvx']
(48842, 15)
Adult columns: ['age' 'workclass' 'fnlwgt' 'education' 'education-num' 'marital-status'
 'occupation' 'relationship' 'race' 'sex' 'capital-gain' 'capital-loss'
 'hours-per-week' 'native-country' 'income']
(8124, 23)
mushroom columns: ['cap-shape' 'cap-surface' 'cap-color' 'bruises' 'odor' 'gill-attachment'
 'gill-spacing' 'gill-size' 'gill-color' 'stalk-shape' 'stalk-root'
 'stalk-surface-above-ring' 'stalk-surface-below-ring'
 'stalk-color-above-ring' 'stalk-color-below-ring' 'veil-type'
 'veil-color' 'ring-number' 'ring-type' 'spore-print-color' 'population'
 'habitat' 'poisonous']


,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,...,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat,poisonous
0,x,s,n,t,p,f,c,n,k,e,...,w,w,p,w,o,p,k,s,u,p
1,x,s,y,t,a,f,c,b,k,e,...,w,w,p,w,o,p,n,n,g,e
2,b,s,w,t,l,f,c,b,n,e,...,w,w,p,w,o,p,n,n,m,e
3,x,y,w,t,p,f,c,n,n,e,...,w,w,p,w,o,p,k,s,u,p
4,x,s,g,f,n,f,w,b,k,t,...,w,w,p,w,o,e,n,a,g,e


In [5]:
for column in mushroom_df.columns.tolist():
    print(f"==={column}===")
    print(mushroom_df[column].value_counts(dropna=False))

===cap-shape===
x    3656
f    3152
k     828
b     452
s      32
c       4
Name: cap-shape, dtype: int64
===cap-surface===
y    3244
s    2556
f    2320
g       4
Name: cap-surface, dtype: int64
===cap-color===
n    2284
g    1840
e    1500
y    1072
w    1040
b     168
p     144
c      44
u      16
r      16
Name: cap-color, dtype: int64
===bruises===
f    4748
t    3376
Name: bruises, dtype: int64
===odor===
n    3528
f    2160
y     576
s     576
a     400
l     400
p     256
c     192
m      36
Name: odor, dtype: int64
===gill-attachment===
f    7914
a     210
Name: gill-attachment, dtype: int64
===gill-spacing===
c    6812
w    1312
Name: gill-spacing, dtype: int64
===gill-size===
b    5612
n    2512
Name: gill-size, dtype: int64
===gill-color===
b    1728
p    1492
w    1202
n    1048
g     752
h     732
u     492
k     408
e      96
y      86
o      64
r      24
Name: gill-color, dtype: int64
===stalk-shape===
t    4608
e    3516
Name: stalk-shape, dtype: int64
===stalk-root===

In [6]:
def bin_numeric_features(df, target_col, n_bins=5, strategy="quantile"):
    binned_df = df.copy()

    for col_name in binned_df.columns:
        if col_name == target_col:
            continue

        series = binned_df[col_name]
        if not pd.api.types.is_numeric_dtype(series):
            continue

        non_null = series.dropna()
        if non_null.nunique() <= 1:
            continue

        if strategy == "quantile":
            # qcut can fail when many duplicated values exist; fallback to cut.
            try:
                binned = pd.qcut(series, q=n_bins, labels=False, duplicates="drop")
            except ValueError:
                binned = pd.cut(series, bins=n_bins, labels=False, include_lowest=True)
        elif strategy == "uniform":
            binned = pd.cut(series, bins=n_bins, labels=False, include_lowest=True)
        else:
            raise ValueError("strategy must be 'quantile' or 'uniform'")

        binned_df[col_name] = binned

    return binned_df


def encode_dataframe_to_int_numpy(df, target_col, random_state=42):
    cols = df.columns.to_numpy(dtype=object)
    working_df = df.copy()
    rng = np.random.default_rng(random_state)

    # Drop feature columns that are entirely missing.
    drop_cols = []
    for col_name in cols:
        if col_name == target_col:
            continue
        if working_df[col_name].isna().all():
            drop_cols.append(col_name)
    if drop_cols:
        working_df = working_df.drop(columns=drop_cols)
        cols = working_df.columns.to_numpy(dtype=object)

    # Impute missing values only in features by sampling from each feature's
    # empirical distribution in observed (non-missing) values.
    for col_name in cols:
        if col_name == target_col:
            continue
        series = working_df[col_name]
        missing_mask = series.isna()
        if not missing_mask.any():
            continue

        observed = series[~missing_mask]
        if observed.shape[0] == 0:
            # Entirely-missing columns are dropped above.
            continue

        value_probs = observed.value_counts(dropna=True, normalize=True)
        sampled_values = rng.choice(
            value_probs.index.to_numpy(dtype=object),
            size=int(missing_mask.sum()),
            p=value_probs.to_numpy(dtype=np.float64),
        )
        working_df.loc[missing_mask, col_name] = sampled_values

    encoded = np.empty((working_df.shape[0], working_df.shape[1]), dtype=np.int64)
    for col_idx, col_name in enumerate(cols):
        series = working_df[col_name]
        if pd.api.types.is_numeric_dtype(series):
            encoded[:, col_idx] = pd.to_numeric(series, errors="coerce").fillna(0).to_numpy(dtype=np.int64)
        else:
            codes, _ = pd.factorize(series.astype(str), sort=True)
            encoded[:, col_idx] = codes.astype(np.int64)

    target_idx = int(np.where(cols == target_col)[0][0])
    feature_indices = np.delete(np.arange(cols.shape[0], dtype=np.int64), target_idx)
    feature_names = cols[feature_indices]

    dataset = np.concatenate(
        [encoded[:, feature_indices], encoded[:, target_idx:target_idx + 1]],
        axis=1,
    )
    return dataset, feature_names


def create_kfold_datasets_multiprocess(
    dataset,
    k=10,
    test_ratio=0.1,
    sample_size_ratio=1.0,
    random_state=42,
    process_workers=None,
):
    data = np.asarray(dataset, dtype=np.int64)
    y = data[:, -1]

    rng = np.random.default_rng(random_state)
    class_values = np.unique(y)

    sampled_indices = np.empty(0, dtype=np.int64)
    for class_value in class_values:
        class_idx = np.where(y == class_value)[0]
        class_idx = rng.permutation(class_idx)
        if sample_size_ratio >= 1.0:
            take_count = class_idx.shape[0]
        else:
            take_count = max(1, int(class_idx.shape[0] * sample_size_ratio))
        sampled_indices = np.concatenate([sampled_indices, class_idx[:take_count]])

    sampled_indices = rng.permutation(sampled_indices)
    sampled_data = data[sampled_indices]
    y_sampled = sampled_data[:, -1]

    # 1) Build one fixed stratified test split.
    fixed_test_indices = np.empty(0, dtype=np.int64)
    trainval_indices = np.empty(0, dtype=np.int64)
    sampled_all_idx = np.arange(sampled_data.shape[0], dtype=np.int64)

    for class_value in np.unique(y_sampled):
        class_idx = sampled_all_idx[y_sampled == class_value]
        class_idx = rng.permutation(class_idx)
        test_count = int(class_idx.shape[0] * test_ratio)
        test_count = max(1, test_count)
        test_count = min(test_count, class_idx.shape[0] - 1)
        fixed_test_indices = np.concatenate([fixed_test_indices, class_idx[:test_count]])
        trainval_indices = np.concatenate([trainval_indices, class_idx[test_count:]])

    fixed_test_indices = rng.permutation(fixed_test_indices)
    trainval_indices = rng.permutation(trainval_indices)
    fixed_test_data = sampled_data[fixed_test_indices]
    trainval_data = sampled_data[trainval_indices]
    y_trainval = trainval_data[:, -1]

    workers = process_workers
    if workers is None:
        workers = max(1, min(k, (os.cpu_count() or 1) - 1))

    task_args = np.empty(class_values.shape[0], dtype=object)
    for i, class_value in enumerate(class_values):
        class_local_idx = np.where(y_trainval == class_value)[0]
        task_args[i] = (int(class_value), class_local_idx, int(k), int(random_state + 1000 + i))

    with ProcessPoolExecutor(max_workers=workers) as executor:
        class_chunks = tuple(executor.map(build_class_folds_worker, task_args))

    fold_validation_indices = np.empty(k, dtype=object)
    for i in range(k):
        fold_validation_indices[i] = np.empty(0, dtype=np.int64)

    for _, chunks in class_chunks:
        for fold_idx in range(k):
            fold_validation_indices[fold_idx] = np.concatenate([fold_validation_indices[fold_idx], chunks[fold_idx]])

    for fold_idx in range(k):
        fold_validation_indices[fold_idx] = rng.permutation(fold_validation_indices[fold_idx])

    all_indices = np.arange(trainval_data.shape[0], dtype=np.int64)
    fold_records = np.empty(k, dtype=object)

    for fold_idx in range(k):
        val_idx = fold_validation_indices[fold_idx]
        train_mask = np.ones(trainval_data.shape[0], dtype=bool)
        train_mask[val_idx] = False
        train_idx = all_indices[train_mask]

        fold_records[fold_idx] = {
            "fold": fold_idx + 1,
            "train": trainval_data[train_idx],
            "validation": trainval_data[val_idx],
            "test": fixed_test_data,
        }

    return fold_records


letter_df_binned = bin_numeric_features(
    letter_df,
    target_col="lettr",
    n_bins=5,
    strategy="quantile",
)

adult_df_binned = bin_numeric_features(
    adult_df,
    target_col="income",
    n_bins=5,
    strategy="quantile",
)

letter_dataset, letter_dataset_feature_names = encode_dataframe_to_int_numpy(letter_df_binned, target_col="lettr")
adult_dataset, adult_dataset_feature_names = encode_dataframe_to_int_numpy(adult_df_binned, target_col="income")
mushroom_dataset, mushroom_dataset_feature_names = encode_dataframe_to_int_numpy(mushroom_df, target_col="poisonous")

print("Applied binning to letter dataset: n_bins=8, strategy='quantile'")
print(letter_dataset_feature_names)
print(adult_dataset_feature_names)
print(mushroom_dataset_feature_names)

letter_fold_datasets = create_kfold_datasets_multiprocess(
    letter_dataset,
    k=10,
    test_ratio=0.1,
    sample_size_ratio=1,
    random_state=42,
)

adult_fold_datasets = create_kfold_datasets_multiprocess(
    adult_dataset,
    k=10,
    test_ratio=0.1,
    sample_size_ratio=1,
    random_state=42,
)

mushroom_fold_datasets = create_kfold_datasets_multiprocess(
    mushroom_dataset,
    k=10,
    test_ratio=0.1,
    sample_size_ratio=1,
    random_state=42,
)

dataset_names = ["letter", "adult", "mushroom"]
original_datasets = [letter_dataset, adult_dataset, mushroom_dataset]
folded_datasets = [letter_fold_datasets, adult_fold_datasets, mushroom_fold_datasets]


for dataset_name, original_dataset, folded_dataset in zip(dataset_names, original_datasets, folded_datasets):
    target_categories = np.unique(original_dataset[:, -1])
    missing_count = int(np.isnan(original_dataset.astype(np.float64)).sum())
    print(f"dataset: {dataset_name}")
    print("dataset shape:", original_dataset.shape)
    print("target categories:", target_categories.tolist())
    print("missing values after preprocessing:", missing_count)
    print("has missing values:", missing_count > 0)
    print("dataset preview (first 5 rows):")
    print(original_dataset[:5])
    print("validation size:", folded_dataset[0]["validation"].shape[0])
    for fold in folded_dataset:
        print(
            f"fold {fold['fold']}: train={fold['train'].shape[0]}, "
            f"test={fold['test'].shape[0]}, validation={fold['validation'].shape[0]}"
        )
    print("\n")



Applied binning to letter dataset: n_bins=8, strategy='quantile'
['x-box' 'y-box' 'width' 'high' 'onpix' 'x-bar' 'y-bar' 'x2bar' 'y2bar'
 'xybar' 'x2ybr' 'xy2br' 'x-ege' 'xegvy' 'y-ege' 'yegvx']
['age' 'workclass' 'fnlwgt' 'education' 'education-num' 'marital-status'
 'occupation' 'relationship' 'race' 'sex' 'capital-gain' 'capital-loss'
 'hours-per-week' 'native-country']
['cap-shape' 'cap-surface' 'cap-color' 'bruises' 'odor' 'gill-attachment'
 'gill-spacing' 'gill-size' 'gill-color' 'stalk-shape' 'stalk-root'
 'stalk-surface-above-ring' 'stalk-surface-below-ring'
 'stalk-color-above-ring' 'stalk-color-below-ring' 'veil-type'
 'veil-color' 'ring-number' 'ring-type' 'spore-print-color' 'population'
 'habitat']
dataset: letter
dataset shape: (20000, 17)
target categories: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]
missing values after preprocessing: 0
has missing values: False
dataset preview (first 5 rows):
[[ 0  2  0  1  0  2  4  0 

# Task3: Train and evaluate all cases 

In [7]:
import time

def _score_prepruning_fold(
    fold_record,
    feature_names,
    min_sample_size,
    max_depth,
    thread_workers=8,
):
    train_data = np.asarray(fold_record["train"], dtype=np.int64)
    test_data = np.asarray(fold_record["test"], dtype=np.int64)
    validation_data = np.asarray(fold_record["validation"], dtype=np.int64)

    default_label = majority_label(train_data[:, -1])
    tree = build_tree(
        train_data,
        np.asarray(feature_names, dtype=object),
        min_sample_size=int(min_sample_size),
        max_depth=max_depth,
        thread_workers=thread_workers,
    )

    val_pred = predict_batch(tree, feature_names, validation_data, default_label)
    val_acc = accuracy_np(validation_data[:, -1], val_pred)

    test_pred = predict_batch(tree, feature_names, test_data, default_label)
    test_acc = accuracy_np(test_data[:, -1], test_pred)

    return {
        "fold": fold_record,
        "validation_accuracy": float(val_acc),
        "test_accuracy": float(test_acc),
        "leaf_count": int(count_leaf_nodes(tree)),
        "depth": int(tree_depth(tree)),
    }


def run_prepruning_all_folds(
    fold_datasets,
    feature_names,
    min_sample_sizes,
    max_depths,
    thread_workers=8,
):
    t0 = time.perf_counter()
    grid = [(int(mss), mxd) for mss in min_sample_sizes for mxd in max_depths]
    grid_total = len(grid)
    if grid_total == 0:
        raise ValueError("min_sample_sizes and max_depths must be non-empty")

    best = {
        "min_sample_size": None,
        "max_depth": None,
        "mean_validation_accuracy": -1.0,
        "mean_leaf_count": float("inf"),
        "per_fold": [],
        "validation_accuracies": np.array([], dtype=np.float64),
        "test_accuracies": np.array([], dtype=np.float64),
    }

    for k, (mss, mxd) in enumerate(grid, start=1):
        per_fold = []
        val_accs = []
        test_accs = []
        leaf_counts = []
        for fold_record in fold_datasets:
            result = _score_prepruning_fold(
                fold_record,
                feature_names,
                min_sample_size=mss,
                max_depth=mxd,
                thread_workers=thread_workers,
            )
            per_fold.append(result)
            val_accs.append(result["validation_accuracy"])
            test_accs.append(result["test_accuracy"])
            leaf_counts.append(result["leaf_count"])

        val_accs = np.asarray(val_accs, dtype=np.float64)
        test_accs = np.asarray(test_accs, dtype=np.float64)
        mean_val = float(val_accs.mean()) if val_accs.size else 0.0
        mean_leaf = float(np.mean(leaf_counts)) if leaf_counts else float("inf")

        is_better = (
            mean_val > best["mean_validation_accuracy"]
            or (mean_val == best["mean_validation_accuracy"] and mean_leaf < best["mean_leaf_count"])
        )
        if is_better:
            best.update(
                min_sample_size=mss,
                max_depth=mxd,
                mean_validation_accuracy=mean_val,
                mean_leaf_count=mean_leaf,
                per_fold=per_fold,
                validation_accuracies=val_accs,
                test_accuracies=test_accs,
            )

        mean_test_cfg = float(test_accs.mean()) if test_accs.size else 0.0
        print(
            f"grid {k}/{grid_total} | mss={mss} max_depth={mxd} "
            f"| mean val={mean_val:.4f} mean test={mean_test_cfg:.4f} "
            f"| best mss={best['min_sample_size']} max_depth={best['max_depth']} "
            f"best mean val={best['mean_validation_accuracy']:.4f}",
            flush=True,
        )

    elapsed = time.perf_counter() - t0
    mean_test = float(best["test_accuracies"].mean()) if best["test_accuracies"].size else 0.0
    print(
        "grid search done in "
        f"{elapsed:.1f}s | best mss={best['min_sample_size']} "
        f"max_depth={best['max_depth']} | mean val={best['mean_validation_accuracy']:.4f} "
        f"mean test={mean_test:.4f}",
        flush=True,
    )

    test_accs = best["test_accuracies"]
    val_accs = best["validation_accuracies"]
    return {
        "best_min_sample_size": best["min_sample_size"],
        "best_max_depth": best["max_depth"],
        "per_fold": best["per_fold"],
        "mean_validation_accuracy_for_selection": best["mean_validation_accuracy"],
        "mean_test_accuracy": float(test_accs.mean()) if test_accs.size else 0.0,
        "std_test_accuracy": float(test_accs.std(ddof=1)) if test_accs.size > 1 else 0.0,
        "test_accuracies": test_accs,
        "validation_accuracies": val_accs,
    }

# Task4-1: Stump vs unpruned vs post-pruned

- **Stump:** `max_depth=1`
- **Unpruned:** `min_sample_size=1`, `max_depth=None`
- **Pruned:** pre-tuning, grid search on min_sample_size and max_depth

In [8]:
# Per-fold checkpoints: metrics JSON + pickled trees (atomic writes).
# Uses THREE_MODEL_CHECKPOINT_ROOT from the imports cell; pass a different checkpoint_dir to override.

def _json_safe_record(row):
    out = {}
    for key, value in row.items():
        if value is None:
            out[key] = None
        elif isinstance(value, float) and value != value:
            out[key] = None
        elif hasattr(value, "item"):
            out[key] = value.item()
        else:
            out[key] = value
    return out


def save_fold_three_model_checkpoint(
    checkpoint_root,
    dataset_name,
    fold_id,
    records,
    trees,
    feature_names,
    default_label,
):
    """Write fold_{id}_records.json and fold_{id}_trees.pkl (atomic rename)."""
    out_dir = Path(checkpoint_root) / str(dataset_name)
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = f"fold_{int(fold_id):03d}"
    json_safe = [_json_safe_record(r) for r in records]

    tmp_json = out_dir / f"{stem}_records.json.tmp"
    final_json = out_dir / f"{stem}_records.json"
    with open(tmp_json, "w", encoding="utf-8") as f:
        json.dump(json_safe, f, indent=2)
    os.replace(tmp_json, final_json)

    payload = {
        "fold": int(fold_id),
        "dataset": str(dataset_name),
        "trees": trees,
        "feature_names": np.asarray(feature_names, dtype=object),
        "default_label": int(default_label),
    }
    tmp_pkl = out_dir / f"{stem}_trees.pkl.tmp"
    final_pkl = out_dir / f"{stem}_trees.pkl"
    with open(tmp_pkl, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp_pkl, final_pkl)


def load_fold_three_model_checkpoint(checkpoint_root, dataset_name, fold_id):
    """Load metrics JSON and trees pickle from a prior run."""
    base = Path(checkpoint_root) / str(dataset_name)
    stem = f"fold_{int(fold_id):03d}"
    json_path = base / f"{stem}_records.json"
    pkl_path = base / f"{stem}_trees.pkl"
    with open(json_path, "r", encoding="utf-8") as f:
        records = json.load(f)
    with open(pkl_path, "rb") as f:
        blob = pickle.load(f)
    return {"records": records, **blob}


# Reload example (after a run has written checkpoints):
# loaded = load_fold_three_model_checkpoint(THREE_MODEL_CHECKPOINT_ROOT, "mushroom", fold_id=1)
# predict_batch(loaded["trees"]["pre_pruned"], loaded["feature_names"], some_array, loaded["default_label"])

In [9]:
# helpers for stump vs unpruned vs pre-pruned (no post-pruning)

def _fit_and_score_case(
    train_data,
    eval_data,  # validation set
    feature_names,
    min_sample_size,
    max_depth,
    thread_workers=8,
):
    default_label = majority_label(train_data[:, -1])
    tree = build_tree(
        train_data,
        np.asarray(feature_names, dtype=object),
        min_sample_size=int(min_sample_size),
        max_depth=max_depth,
        thread_workers=thread_workers,
    )
    eval_pred = predict_batch(tree, feature_names, eval_data, default_label)
    eval_acc = accuracy_np(eval_data[:, -1], eval_pred)
    return {
        "tree": tree,
        "validation_accuracy": float(eval_acc),
        "leaf_count": int(count_leaf_nodes(tree)),
        "depth": int(tree_depth(tree)),
    }



In [10]:
def max_unpruned_depth_across_folds(
    fold_datasets,
    feature_names,
    thread_workers=8,
):
    max_depth = 0
    for fold_record in fold_datasets:
        train_data = np.asarray(fold_record["train"], dtype=np.int64)
        tree = build_tree(
            train_data,
            np.asarray(feature_names, dtype=object),
            min_sample_size=1,
            max_depth=None,
            thread_workers=thread_workers,
        )
        depth = int(tree_depth(tree))
        if depth > max_depth:
            max_depth = depth
    return max_depth


def evaluate_three_models_one_fold_preselected(
    fold_record,
    feature_names,
    prepruned_choice,
    thread_workers=8,
    checkpoint_dir=None,
    dataset_name=None,
):
    if prepruned_choice is None:
        raise ValueError("prepruned_choice is required (use run_prepruning_all_folds)")

    train_data = np.asarray(fold_record["train"], dtype=np.int64)
    test_data = np.asarray(fold_record["test"], dtype=np.int64)          # final unbiased metric
    val_data = np.asarray(fold_record["validation"], dtype=np.int64)     # used for validation report
    fold_id = int(fold_record["fold"])

    # 1) Stump: max_depth=1, report on validation
    stump = _fit_and_score_case(
        train_data=train_data,
        eval_data=val_data,
        feature_names=feature_names,
        min_sample_size=1,
        max_depth=0,
        thread_workers=thread_workers,
    )

    # 2) Unpruned: min_sample_size=1, max_depth=None, report on validation
    unpruned = _fit_and_score_case(
        train_data=train_data,
        eval_data=val_data,
        feature_names=feature_names,
        min_sample_size=1,
        max_depth=None,
        thread_workers=thread_workers,
    )

    # unbiased final test metrics for fixed-model baselines
    default_label = majority_label(train_data[:, -1])
    stump_test_pred = predict_batch(stump["tree"], feature_names, test_data, default_label)
    stump_test_acc = float(accuracy_np(test_data[:, -1], stump_test_pred))
    unpruned_test_pred = predict_batch(unpruned["tree"], feature_names, test_data, default_label)
    unpruned_test_acc = float(accuracy_np(test_data[:, -1], unpruned_test_pred))

    # 3) Pre-pruned: chosen by mean validation accuracy across folds
    pre_mss = int(prepruned_choice["min_sample_size"])
    pre_md = prepruned_choice["max_depth"]
    selection_val = float(prepruned_choice.get("mean_validation_accuracy", np.nan))

    pre_tree = build_tree(
        train_data,
        np.asarray(feature_names, dtype=object),
        min_sample_size=pre_mss,
        max_depth=pre_md,
        thread_workers=thread_workers,
    )
    pre_val_pred = predict_batch(pre_tree, feature_names, val_data, default_label)
    pre_val_acc = float(accuracy_np(val_data[:, -1], pre_val_pred))
    pre_test_pred = predict_batch(pre_tree, feature_names, test_data, default_label)
    pre_test_acc = float(accuracy_np(test_data[:, -1], pre_test_pred))

    rows = [
        {
            "fold": fold_id,
            "model": "stump",
            "validation_accuracy": stump["validation_accuracy"],
            "leaf_count": stump["leaf_count"],
            "depth": stump["depth"],
            "best_min_sample_size": 1,
            "best_max_depth": 1,
            "selection_validation_accuracy": np.nan,
            "test_accuracy": stump_test_acc,
        },
        {
            "fold": fold_id,
            "model": "unpruned",
            "validation_accuracy": unpruned["validation_accuracy"],
            "leaf_count": unpruned["leaf_count"],
            "depth": unpruned["depth"],
            "best_min_sample_size": 1,
            "best_max_depth": None,
            "selection_validation_accuracy": np.nan,
            "test_accuracy": unpruned_test_acc,
        },
        {
            "fold": fold_id,
            "model": "pre_pruned",
            "validation_accuracy": pre_val_acc,
            "leaf_count": int(count_leaf_nodes(pre_tree)),
            "depth": int(tree_depth(pre_tree)),
            "best_min_sample_size": pre_mss,
            "best_max_depth": pre_md,
            "selection_validation_accuracy": selection_val,
            "test_accuracy": pre_test_acc,
        },
    ]

    if checkpoint_dir is not None and dataset_name is not None:
        save_fold_three_model_checkpoint(
            checkpoint_dir,
            dataset_name,
            fold_id,
            rows,
            {
                "stump": stump["tree"],
                "unpruned": unpruned["tree"],
                "pre_pruned": pre_tree,
            },
            feature_names,
            default_label,
        )

    return rows


In [11]:
# strict nested CV (outer test folds + inner tuning) and summary

from IPython.display import display

STUMP_MAX_DEPTH = 1
OUTER_K = 10
INNER_K = 5
RANDOM_STATE = 42
min_sample_sizes = list(range(1, 6, 2))


def make_stratified_folds(y, k, random_state):
    y = np.asarray(y, dtype=np.int64)
    rng = np.random.default_rng(int(random_state))
    per_fold = [np.empty(0, dtype=np.int64) for _ in range(int(k))]

    for class_value in np.unique(y):
        class_idx = np.where(y == class_value)[0]
        class_idx = rng.permutation(class_idx)
        chunks = np.array_split(class_idx, int(k))
        for fold_idx, chunk in enumerate(chunks):
            if chunk.size:
                per_fold[fold_idx] = np.concatenate([per_fold[fold_idx], chunk.astype(np.int64)])

    for fold_idx in range(int(k)):
        per_fold[fold_idx] = rng.permutation(per_fold[fold_idx])

    return per_fold


def get_max_depth_cap(dataset, feature_names, thread_workers=8):
    full_unpruned = build_tree(
        np.asarray(dataset, dtype=np.int64),
        np.asarray(feature_names, dtype=object),
        min_sample_size=1,
        max_depth=None,
        thread_workers=thread_workers,
    )
    return int(max(0, tree_depth(full_unpruned) - 1))


def tune_prepruned_inner_cv(train_data, feature_names, min_sample_sizes, max_depths, inner_k=5, random_state=42, thread_workers=8):
    train_data = np.asarray(train_data, dtype=np.int64)
    y = train_data[:, -1]
    inner_val_folds = make_stratified_folds(y, k=inner_k, random_state=random_state)
    all_idx = np.arange(train_data.shape[0], dtype=np.int64)

    best = {
        "min_sample_size": None,
        "max_depth": None,
        "mean_validation_accuracy": -1.0,
        "mean_leaf_count": float("inf"),
    }

    grid = [(int(mss), int(md)) for mss in min_sample_sizes for md in max_depths]
    for mss, md in grid:
        val_accs = []
        leaf_counts = []

        for val_idx in inner_val_folds:
            train_mask = np.ones(train_data.shape[0], dtype=bool)
            train_mask[val_idx] = False
            inner_train = train_data[all_idx[train_mask]]
            inner_val = train_data[val_idx]

            default_label = majority_label(inner_train[:, -1])
            tree = build_tree(
                inner_train,
                np.asarray(feature_names, dtype=object),
                min_sample_size=mss,
                max_depth=md,
                thread_workers=thread_workers,
            )
            val_pred = predict_batch(tree, feature_names, inner_val, default_label)
            val_accs.append(float(accuracy_np(inner_val[:, -1], val_pred)))
            leaf_counts.append(int(count_leaf_nodes(tree)))

        mean_val = float(np.mean(val_accs)) if val_accs else 0.0
        mean_leaf = float(np.mean(leaf_counts)) if leaf_counts else float("inf")

        if (mean_val > best["mean_validation_accuracy"]) or (
            mean_val == best["mean_validation_accuracy"] and mean_leaf < best["mean_leaf_count"]
        ):
            best.update(
                min_sample_size=mss,
                max_depth=md,
                mean_validation_accuracy=mean_val,
                mean_leaf_count=mean_leaf,
            )

    return best


def run_nested_three_models_one_dataset(
    dataset_name,
    dataset,
    feature_names,
    min_sample_sizes,
    outer_k=10,
    inner_k=5,
    random_state=42,
    thread_workers=8,
):
    data = np.asarray(dataset, dtype=np.int64)
    feature_names = np.asarray(feature_names, dtype=object)

    # Use a fixed, data-independent depth grid for strict nesting.
    max_depth_cap = int(feature_names.shape[0])
    max_depths = list(range(0, max_depth_cap + 1))
    print(f"\n=== {dataset_name} ===")
    print("nested grid max_depth:", max_depths)

    outer_test_folds = make_stratified_folds(data[:, -1], k=outer_k, random_state=random_state)
    all_idx = np.arange(data.shape[0], dtype=np.int64)

    rows = []
    pre_choice_rows = []
    fold_artifacts = []

    for fold_id, test_idx in enumerate(outer_test_folds, start=1):
        train_mask = np.ones(data.shape[0], dtype=bool)
        train_mask[test_idx] = False
        train_idx = all_idx[train_mask]

        outer_train = data[train_idx]
        outer_test = data[test_idx]

        pre_choice = tune_prepruned_inner_cv(
            outer_train,
            feature_names,
            min_sample_sizes=min_sample_sizes,
            max_depths=max_depths,
            inner_k=inner_k,
            random_state=random_state + 1000 + fold_id,
            thread_workers=thread_workers,
        )

        pre_mss = int(pre_choice["min_sample_size"])
        pre_md = int(pre_choice["max_depth"])
        default_label = majority_label(outer_train[:, -1])

        models = {
            "stump": build_tree(outer_train, feature_names, min_sample_size=1, max_depth=STUMP_MAX_DEPTH, thread_workers=thread_workers),
            "unpruned": build_tree(outer_train, feature_names, min_sample_size=1, max_depth=None, thread_workers=thread_workers),
            "pre_pruned": build_tree(outer_train, feature_names, min_sample_size=pre_mss, max_depth=pre_md, thread_workers=thread_workers),
        }

        preds = {}
        for model_name, tree in models.items():
            train_pred = predict_batch(tree, feature_names, outer_train, default_label)
            test_pred = predict_batch(tree, feature_names, outer_test, default_label)
            preds[model_name] = test_pred

            rows.append(
                {
                    "dataset": dataset_name,
                    "fold": fold_id,
                    "model": model_name,
                    "train_accuracy": float(accuracy_np(outer_train[:, -1], train_pred)),
                    "test_accuracy": float(accuracy_np(outer_test[:, -1], test_pred)),
                    "leaf_count": int(count_leaf_nodes(tree)),
                    "depth": int(tree_depth(tree)),
                    "best_min_sample_size": pre_mss if model_name == "pre_pruned" else np.nan,
                    "best_max_depth": pre_md if model_name == "pre_pruned" else np.nan,
                    "selection_validation_accuracy": float(pre_choice["mean_validation_accuracy"]) if model_name == "pre_pruned" else np.nan,
                }
            )

        pre_choice_rows.append(
            {
                "dataset": dataset_name,
                "fold": fold_id,
                "best_min_sample_size": pre_mss,
                "best_max_depth": pre_md,
                "selection_validation_accuracy": float(pre_choice["mean_validation_accuracy"]),
            }
        )
        fold_artifacts.append(
            {
                "dataset": dataset_name,
                "fold": fold_id,
                "y_test": outer_test[:, -1].copy(),
                "preds": preds,
            }
        )

        print(
            f"outer fold {fold_id}/{outer_k} | tuned pre-pruned: mss={pre_mss}, max_depth={pre_md}, "
            f"inner mean val={pre_choice['mean_validation_accuracy']:.4f}"
        )

    return rows, pre_choice_rows, fold_artifacts


configs = [
    ("letter", letter_dataset, letter_dataset_feature_names),
    ("adult", adult_dataset, adult_dataset_feature_names),
    ("mushroom", mushroom_dataset, mushroom_dataset_feature_names),
]

all_rows = []
all_pre_choices = []
NESTED_TEST_PREDICTIONS = []

for dataset_name, dataset, feature_names in configs:
    ds_rows, ds_choices, ds_artifacts = run_nested_three_models_one_dataset(
        dataset_name=dataset_name,
        dataset=dataset,
        feature_names=feature_names,
        min_sample_sizes=min_sample_sizes,
        outer_k=OUTER_K,
        inner_k=INNER_K,
        random_state=RANDOM_STATE,
        thread_workers=8,
    )
    all_rows.extend(ds_rows)
    all_pre_choices.extend(ds_choices)
    NESTED_TEST_PREDICTIONS.extend(ds_artifacts)

results_df = pd.DataFrame(all_rows)
results_df["depth_aligned_to_max_depth"] = results_df["depth"] - 1

print("\n=== Nested CV per-fold test results ===")
display(results_df.sort_values(["dataset", "fold", "model"]).reset_index(drop=True))

summary_df = (
    results_df
    .groupby(["dataset", "model"], as_index=False)
    .agg(
        mean_train_accuracy=("train_accuracy", "mean"),
        std_train_accuracy=("train_accuracy", "std"),
        mean_test_accuracy=("test_accuracy", "mean"),
        std_test_accuracy=("test_accuracy", "std"),
        mean_leaf_count=("leaf_count", "mean"),
        mean_depth=("depth_aligned_to_max_depth", "mean"),
    )
    .sort_values(["dataset", "mean_test_accuracy"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\n=== Nested CV summary (mean +/- std test accuracy) ===")
display(summary_df)

pre_choices = (
    pd.DataFrame(all_pre_choices)
    .groupby(["dataset", "best_min_sample_size", "best_max_depth"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["dataset", "count", "best_max_depth", "best_min_sample_size"], ascending=[True, False, True, True])
    .reset_index(drop=True)
)

print("\n=== Nested pre-pruned hyperparameter choices (outer folds) ===")
display(pre_choices)

best_prepruned_by_dataset = (
    pre_choices
    .drop_duplicates(subset=["dataset"], keep="first")
    .rename(columns={"count": "selected_in_outer_folds"})
    .sort_values(["dataset"])
    .reset_index(drop=True)
)

print("\n=== Final best pre-pruned hyperparameters (one per dataset) ===")
display(best_prepruned_by_dataset)


=== letter ===
nested grid max_depth: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
outer fold 1/10 | tuned pre-pruned: mss=1, max_depth=6, inner mean val=0.8038
outer fold 2/10 | tuned pre-pruned: mss=1, max_depth=10, inner mean val=0.8021
outer fold 3/10 | tuned pre-pruned: mss=1, max_depth=7, inner mean val=0.8002
outer fold 4/10 | tuned pre-pruned: mss=1, max_depth=7, inner mean val=0.8073
outer fold 5/10 | tuned pre-pruned: mss=1, max_depth=7, inner mean val=0.8032
outer fold 6/10 | tuned pre-pruned: mss=1, max_depth=7, inner mean val=0.8014
outer fold 7/10 | tuned pre-pruned: mss=1, max_depth=7, inner mean val=0.8018
outer fold 8/10 | tuned pre-pruned: mss=1, max_depth=10, inner mean val=0.7983
outer fold 9/10 | tuned pre-pruned: mss=1, max_depth=8, inner mean val=0.8043
outer fold 10/10 | tuned pre-pruned: mss=1, max_depth=7, inner mean val=0.8043

=== adult ===
nested grid max_depth: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
outer fold 1/10 | tuned pre-pr

KeyboardInterrupt: 

In [ ]:
# Retrain pre-pruned trees on the full preprocessed dataset (best hyperparameters from grid search).

best_prepruned_hparams = {
    "adult": {"min_sample_size": 3, "max_depth": 3},
    "letter": {"min_sample_size": 1, "max_depth": 7},
    "mushroom": {"min_sample_size": 1, "max_depth": 4},
}

full_data_and_names = {
    "letter": (letter_dataset, letter_dataset_feature_names),
    "adult": (adult_dataset, adult_dataset_feature_names),
    "mushroom": (mushroom_dataset, mushroom_dataset_feature_names),
}

final_prepruned_trees = {}
for name, hp in best_prepruned_hparams.items():
    data, feat_names = full_data_and_names[name]
    tree = build_tree(
        data,
        feat_names,
        min_sample_size=int(hp["min_sample_size"]),
        max_depth=int(hp["max_depth"]),
        thread_workers=8,
    )
    final_prepruned_trees[name] = tree
    default_label = majority_label(data[:, -1])
    train_acc = accuracy_np(
        data[:, -1],
        predict_batch(tree, feat_names, data, default_label),
    )
    print(
        f"{name}: mss={hp['min_sample_size']} max_depth={hp['max_depth']} | "
        f"leaves={count_leaf_nodes(tree)} depth={tree_depth(tree)} train_acc={train_acc:.4f}"
    )

# Optional: persist
out_dir = Path.cwd() / "final_full_dataset_trees"
out_dir.mkdir(parents=True, exist_ok=True)
for name, tree in final_prepruned_trees.items():
    with (out_dir / f"{name}_prepruned_full.pkl").open("wb") as f:
        pickle.dump({"tree": tree, "hparams": best_prepruned_hparams[name]}, f)
print("Saved to", out_dir)


## Task4-2: Final paired t-tests from nested outer folds

Predictions are taken from the **strict nested CV** run above: for each outer fold, pre-pruned hyperparameters are tuned only on the outer-train split (inner CV), then all three models are evaluated on that fold's held-out outer test split.

For each model pair and dataset, let $\mathrm{err}(M_1)_i$ and $\mathrm{err}(M_2)_i$ be the fold-$i$ test error rates over the same outer partitions. We compute:

$t = \frac{\overline{d}}{\sqrt{\mathrm{var}(d)/k}}$, where $d_i = \mathrm{err}(M_1)_i - \mathrm{err}(M_2)_i$,

$\mathrm{var}(d)=\frac{1}{k}\sum_{i=1}^{k}\left(d_i-\overline{d}\right)^2$,

with $k-1$ degrees of freedom for the two-tailed paired t-test.

In [ ]:
import math

if "NESTED_TEST_PREDICTIONS" not in globals() or len(NESTED_TEST_PREDICTIONS) == 0:
    raise RuntimeError("Run the nested CV cell first to populate NESTED_TEST_PREDICTIONS.")

try:
    from scipy import stats as _scipy_stats
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False


def _norm_sf_abs(z):
    """Two-tailed p-value under standard normal (fallback if scipy missing)."""
    z = float(abs(z))
    return math.erfc(z / math.sqrt(2.0))


def _paired_t_stat_and_p(diff_values):
    """Return paired t-statistic and two-tailed p-value for fold-wise differences."""
    diffs = np.asarray(diff_values, dtype=np.float64)
    k = int(diffs.size)
    if k < 2:
        return float("nan"), float("nan"), k, float("nan")

    mean_diff = float(diffs.mean())
    # Use population-style variance here to match the report formula.
    var_diff = float(np.mean((diffs - mean_diff) ** 2))
    denom = math.sqrt(var_diff / k) if var_diff > 0.0 else 0.0

    if denom == 0.0:
        # All fold differences are identical; no evidence of difference.
        return float("nan"), 1.0, k, var_diff

    t_stat = mean_diff / denom
    df = k - 1

    if _HAS_SCIPY:
        p_two_tailed = float(2.0 * _scipy_stats.t.sf(abs(t_stat), df=df))
    else:
        p_two_tailed = float(_norm_sf_abs(t_stat))

    return float(t_stat), p_two_tailed, k, var_diff


pairs = [
    ("stump", "pre_pruned"),
    ("stump", "unpruned"),
    ("pre_pruned", "unpruned"),
]

per_fold_rows = []
for fold_artifact in NESTED_TEST_PREDICTIONS:
    dataset_name = fold_artifact["dataset"]
    fold_id = int(fold_artifact["fold"])
    y_te = np.asarray(fold_artifact["y_test"], dtype=np.int64)
    preds = fold_artifact["preds"]

    for a, b in pairs:
        err_a = float(np.mean(preds[a] != y_te))
        err_b = float(np.mean(preds[b] != y_te))
        per_fold_rows.append(
            {
                "dataset": dataset_name,
                "fold": fold_id,
                "model_a": a,
                "model_b": b,
                "err_a": err_a,
                "err_b": err_b,
                "diff_err": err_a - err_b,
            }
        )

pair_df = pd.DataFrame(per_fold_rows)

summary_rows = []
for (dataset_name, model_a, model_b), g in pair_df.groupby(["dataset", "model_a", "model_b"], sort=False):
    t_stat, p_value, k, var_diff = _paired_t_stat_and_p(g["diff_err"].to_numpy())
    summary_rows.append(
        {
            "dataset": dataset_name,
            "model_a": model_a,
            "model_b": model_b,
            "k_folds": int(k),
            "mean_err_model_a": float(g["err_a"].mean()),
            "mean_err_model_b": float(g["err_b"].mean()),
            "mean_diff_err": float(g["diff_err"].mean()),
            "var_diff": float(var_diff) if var_diff == var_diff else np.nan,
            "t_stat": t_stat,
            "p_two_tailed": p_value,
            "df": int(k - 1) if k >= 1 else np.nan,
            "p_method": "student_t" if _HAS_SCIPY else "normal_approx_fallback",
        }
    )

pair_ttest_summary = (
    pd.DataFrame(summary_rows)
    .sort_values(["dataset", "p_two_tailed"], ascending=[True, True])
    .reset_index(drop=True)
)

print("=== Fold-wise error differences for paired t-test ===")
display(pair_df.sort_values(["dataset", "fold", "model_a", "model_b"]).reset_index(drop=True))

print("\n=== Paired t-test summary across outer folds ===")
display(pair_ttest_summary)

## Statistical significance discussion (paired t-test, $\alpha=0.05$)

Using the paired t-test results across 10 outer folds, we compare model error rates (`stump`, `pre_pruned`, `unpruned`) for each dataset. A difference is considered statistically significant when `p < 0.05`.

### 1) Adult dataset
- `stump` vs `pre_pruned`: `p = 8.75e-13 < 0.05` -> **significant**. `pre_pruned` has lower mean error (0.4528 vs 0.4939), so `pre_pruned` is significantly better.
- `pre_pruned` vs `unpruned`: `p = 3.20e-11 < 0.05` -> **significant**. `pre_pruned` has lower mean error (0.4528 vs 0.5173), so `pre_pruned` is significantly better.
- `stump` vs `unpruned`: `p = 1.17e-07 < 0.05` -> **significant**. `stump` has lower mean error (0.4939 vs 0.5173), so `stump` is significantly better.

**Conclusion (Adult):** All three pairwise differences are statistically significant.

### 2) Letter dataset
- `stump` vs `pre_pruned`: `p = 6.27e-19 < 0.05` -> **significant**. `pre_pruned` is much better (0.1834 vs 0.8626).
- `stump` vs `unpruned`: `p = 3.86e-18 < 0.05` -> **significant**. `unpruned` is much better (0.1833 vs 0.8626).
- `pre_pruned` vs `unpruned`: `p = 0.9301 > 0.05` -> **not significant**. Their mean errors are almost identical (0.183381 vs 0.183283).

**Conclusion (Letter):** `stump` is significantly worse than both tree models, while `pre_pruned` and `unpruned` are not significantly different.

### 3) Mushroom dataset
- `stump` vs `pre_pruned`: `p = 1.06e-05 < 0.05` -> **significant**. `pre_pruned` is better (0.0000 vs 0.0148).
- `stump` vs `unpruned`: `p = 1.06e-05 < 0.05` -> **significant**. `unpruned` is better (0.0000 vs 0.0148).
- `pre_pruned` vs `unpruned`: `p = 1.0000 > 0.05` -> **not significant**. Both have zero mean error.

**Conclusion (Mushroom):** `stump` is significantly worse; `pre_pruned` and `unpruned` are statistically indistinguishable.

### Overall conclusion
At significance level `0.05`, most model differences are statistically significant. The only non-significant comparisons are `pre_pruned` vs `unpruned` on the **letter** and **mushroom** datasets, where their performances are essentially the same.

# Final model on whole dataset

In [ ]:
# Full-data CV tuning + final pre-pruned retraining (appended final step).
# Note: this step is for final model training, while unbiased performance comes from nested CV above.

full_data_and_names = {
    "letter": (letter_dataset, letter_dataset_feature_names),
    "adult": (adult_dataset, adult_dataset_feature_names),
    "mushroom": (mushroom_dataset, mushroom_dataset_feature_names),
}

full_tune_random_state = int(globals().get("RANDOM_STATE", 42))
full_tune_inner_k = 5
full_tune_min_sample_sizes = list(globals().get("min_sample_sizes", list(range(1, 6, 2))))

best_prepruned_hparams_full_data = {}
print("=== Full-data CV tuning for pre-pruned model ===")
for name, (data, feat_names) in full_data_and_names.items():
    data = np.asarray(data, dtype=np.int64)
    feat_names = np.asarray(feat_names, dtype=object)

    # Keep the same depth-grid idea used in nested tuning.
    max_depth_cap = int(feat_names.shape[0])
    max_depths = list(range(0, max_depth_cap + 1))

    best = tune_prepruned_inner_cv(
        train_data=data,
        feature_names=feat_names,
        min_sample_sizes=full_tune_min_sample_sizes,
        max_depths=max_depths,
        inner_k=full_tune_inner_k,
        random_state=full_tune_random_state,
        thread_workers=8,
    )

    best_prepruned_hparams_full_data[name] = {
        "min_sample_size": int(best["min_sample_size"]),
        "max_depth": int(best["max_depth"]),
        "cv_mean_validation_accuracy": float(best["mean_validation_accuracy"]),
    }

    print(
        f"{name}: best mss={best_prepruned_hparams_full_data[name]['min_sample_size']} "
        f"max_depth={best_prepruned_hparams_full_data[name]['max_depth']} "
        f"| mean CV val acc={best_prepruned_hparams_full_data[name]['cv_mean_validation_accuracy']:.4f}"
    )

print("\n=== Final selected pre-pruned hyperparameters (full-data CV) ===")
display(pd.DataFrame(best_prepruned_hparams_full_data).T)

final_prepruned_trees_full_data = {}
for name, hp in best_prepruned_hparams_full_data.items():
    data, feat_names = full_data_and_names[name]
    tree = build_tree(
        data,
        feat_names,
        min_sample_size=int(hp["min_sample_size"]),
        max_depth=int(hp["max_depth"]),
        thread_workers=8,
    )
    final_prepruned_trees_full_data[name] = tree

    default_label = majority_label(data[:, -1])
    train_acc = accuracy_np(
        data[:, -1],
        predict_batch(tree, feat_names, data, default_label),
    )
    print(
        f"{name}: mss={hp['min_sample_size']} max_depth={hp['max_depth']} | "
        f"leaves={count_leaf_nodes(tree)} depth={tree_depth(tree)} train_acc={train_acc:.4f}"
    )

# Optional: persist to separate folder to avoid overwriting earlier artifacts.
out_dir = Path.cwd() / "final_full_dataset_trees_tuned"
out_dir.mkdir(parents=True, exist_ok=True)
for name, tree in final_prepruned_trees_full_data.items():
    with (out_dir / f"{name}_prepruned_full.pkl").open("wb") as f:
        pickle.dump({"tree": tree, "hparams": best_prepruned_hparams_full_data[name]}, f)
print("Saved to", out_dir)
